<a href="https://colab.research.google.com/github/palindromeRice/SingleBitLLMs/blob/main/SingleBitLLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 📦 Installing Required Libraries

To begin our experimentation, we install the necessary Python packages:

- `bitnet`: A lightweight package (hypothetical or custom) used for working with **BitNet models** or related utilities. This may contain implementations or wrappers for models quantized to very low-bit precision, such as BitNet b1.58, known for its 1.58-bit weight precision.

- `datasets`: A library by Hugging Face that provides easy access to a wide variety of datasets. We'll use this to load, preprocess, and work with structured datasets in a seamless and efficient manner.


In [1]:
!pip install -q bitnet
!pip install -q datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### 🔧 BitLinear Layer Demo

We initialize a `BitLinear` layer from the `bitnet` package and apply it to a random input tensor. This layer functions like `nn.Linear` but uses low-bit quantized weights, enabling efficient memory usage and faster inference.


In [ ]:
import torch
from bitnet import BitLinear

# Define input
x = torch.randn(10, 1000, 512)

# Initialize BitLinear layer
layer = BitLinear(512, 400)

# Forward pass
y = layer(x)

print(y.shape)


torch.Size([10, 1000, 400])


### 📥 Dataset Loading and Exploration

The SST-2 dataset from the GLUE benchmark is loaded for binary sentiment classification. The training and validation splits are converted into DataFrames for easier inspection. Sample entries are displayed, and the label distribution is printed to check for class balance. This helps ensure a well-understood dataset before model training.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from bitnet import BitLinear
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

raw = load_dataset("glue", "sst2")
train_df = pd.DataFrame(raw["train"])
val_df = pd.DataFrame(raw["validation"])
print("Training samples:")
print(train_df.head())
print("\nValidation samples:")
print(val_df.head())
print("\nLabel distribution (train):")
print(train_df["label"].value_counts())

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Training samples:
                                            sentence  label  idx
0       hide new secretions from the parental units       0    0
1               contains no wit , only labored gags       0    1
2  that loves its characters and communicates som...      1    2
3  remains utterly satisfied to remain the same t...      0    3
4  on the worst revenge-of-the-nerds clichés the ...      0    4

Validation samples:
                                            sentence  label  idx
0    it 's a charming and often affecting journey .       1    0
1                 unflinchingly bleak and desperate       0    1
2  allows us to hope that nolan is poised to emba...      1    2
3  the acting , costumes , music , cinematography...      1    3
4                  it 's slow -- very , very slow .       0    4

Label distribution (train):
label
1    37569
0    29780
Name: count, dtype: int64


### 🧠 Tokenization and Quantized Transformer Model

We load a tokenizer from the BitNet model and ensure it has a padding token defined. The SST-2 dataset is tokenized with a fixed max length and mapped into PyTorch tensors.

Next, we define a custom **Quantized Transformer Classifier**, which includes:
- An embedding layer for input tokens.
- A multi-head self-attention mechanism.
- Feedforward layers using `BitLinear` for low-bit quantized computation.
- Residual connections and layer normalization.
- A classification head for binary sentiment prediction.

This architecture is lightweight and optimized for efficient inference using BitNet's quantized layers.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

def tokenize_fn(ex):
    return tokenizer(ex["sentence"], padding="max_length", truncation=True, max_length=128)

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

class QuantizedTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048, num_labels=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            BitLinear(embed_dim, ff_dim),
            nn.GELU(),
            BitLinear(ff_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids)
        x = x.permute(1, 0, 2)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=~attention_mask.bool())
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        cls = x[0]
        return self.classifier(cls)



Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
vocab_size = len(tokenizer)
print(vocab_size)

128257


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerClassifier(vocab_size=tokenizer.vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)

### Model Architecture Visualisation

In [ ]:
print(model)

QuantizedTransformerClassifier(
  (embed): Embedding(128000, 768)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (ff): Sequential(
    (0): BitLinear(in_features=768, out_features=2048, bias=True)
    (1): GELU(approximate='none')
    (2): BitLinear(in_features=2048, out_features=768, bias=True)
  )
  (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(in_features=768, out_features=2, bias=True)
)


### 🚀 Approach 1: Training a Quantized Transformer on SST-2

In this approach, we build and train a lightweight Transformer-based classifier using low-bit quantized `BitLinear` layers for improved efficiency.

- **Tokenizer Setup**: The BitNet tokenizer is loaded and updated with a padding token if missing. The SST-2 dataset is tokenized and formatted into PyTorch tensors.
- **Model Architecture**: A custom `QuantizedTransformerClassifier` is defined with:
  - Embedding layer
  - Multi-head self-attention
  - Quantized feedforward layers using `BitLinear`
  - Residual connections and layer normalization
  - Final linear classification head
- **Training Loop**: The model is trained for 3 epochs using Adam optimizer and cross-entropy loss. Training is done on GPU if available.
- **Evaluation**: After training, the model is evaluated on the SST-2 validation set. Accuracy and F1-score are computed to measure performance.

This quantized setup balances performance and computational efficiency, making it ideal for resource-constrained environments.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer
from bitnet import BitLinear
from sklearn.metrics import accuracy_score, f1_score

raw = load_dataset("glue", "sst2")

tokenizer = AutoTokenizer.from_pretrained("microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab_size = len(tokenizer)

def tokenize_fn(ex):
    return tokenizer(ex["sentence"], padding="max_length", truncation=True, max_length=128)

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

class QuantizedTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048, num_labels=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            BitLinear(embed_dim, ff_dim),
            nn.GELU(),
            BitLinear(ff_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids)
        x = x.permute(1, 0, 2)
        attn_out, _ = self.attn(x, x, x, key_padding_mask=~attention_mask.bool())
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        cls = x[0]
        return self.classifier(cls)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerClassifier(vocab_size=vocab_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)

for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        mask      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(input_ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} → Train Loss: {total_loss/len(train_loader):.4f}")

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        mask      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device)
        logits = model(input_ids, mask)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
print(f"SST-2 Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1 → Train Loss: 0.5404
Epoch 2 → Train Loss: 0.3803
Epoch 3 → Train Loss: 0.3116
SST-2 Validation → Accuracy: 0.7638, F1: 0.7638


In [3]:
!pip install contractions
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.3/118.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 39.8 MB/s eta 0:00:00


### ⚙️ Approach 2: Enhanced Quantized Transformer with Preprocessing & Custom Attention

This approach builds upon the previous one with added text preprocessing and a custom self-attention mechanism using quantized layers.

- **Text Cleaning**: Each sentence is lowercased, contractions are expanded, emojis are demojized, punctuation is cleaned, and basic negation handling is applied to improve semantic clarity.
- **Tokenization**: The cleaned data is tokenized using the BitNet tokenizer with padding and truncation to a fixed length.
- **Model Architecture**: A custom `QuantizedTransformerClassifier` is defined featuring:
  - Token embeddings with sinusoidal positional encoding.
  - Custom self-attention built entirely using `BitLinear` layers for queries, keys, and values.
  - Quantized feedforward layers and residual normalization.
  - Mean pooling over valid tokens instead of relying on a special CLS token.
- **Training Loop**: Uses the AdamW optimizer with weight decay and a linear learning rate scheduler with warmup steps. Gradient clipping is also applied for stability.
- **Evaluation**: After training for 3 epochs, model performance is assessed using accuracy and weighted F1-score on the SST-2 validation set.




In [5]:
import re
import contractions
import emoji
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from bitnet import BitLinear
from sklearn.metrics import accuracy_score, f1_score

# 1️⃣ LOAD & CLEAN TEXT
raw = load_dataset("glue", "sst2")

def clean_text(example):
    sentence = example["sentence"]
    # fix casing & contractions
    sentence = contractions.fix(sentence.lower())
    # turn emojis into text (optional)
    sentence = emoji.demojize(sentence)
    # keep letters, numbers, spaces, ! and ?
    sentence = re.sub(r"[^a-zA-Z0-9\s!?]", "", sentence)
    # simple negation handling: "not good" → "not_good"
    sentence = re.sub(r"\bnot\s+(\w+)", r"not_\1", sentence)
    # collapse extra spaces
    example["sentence"] = " ".join(sentence.split())
    return example

raw = raw.map(clean_text)

# 2️⃣ TOKENIZE
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab_size = len(tokenizer)

def tokenize_fn(batch):
    return tokenizer(
        batch["sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3️⃣ MODEL DEFINITION
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = pe.unsqueeze(0)  # shape (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, : x.size(1)].to(x.device)

class QuantizedTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048, num_labels=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)
        # simplified BitLinear self-attention
        self.to_q = BitLinear(embed_dim, embed_dim)
        self.to_k = BitLinear(embed_dim, embed_dim)
        self.to_v = BitLinear(embed_dim, embed_dim)
        self.norm1 = nn.LayerNorm(embed_dim)
        # feed-forward
        self.ff = nn.Sequential(
            BitLinear(embed_dim, ff_dim),
            nn.GELU(),
            BitLinear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask):
        x = self.embed(input_ids)          # (B, T, D)
        x = self.pos_enc(x)

        # Self-Attention
        q = self.to_q(x)
        k = self.to_k(x)
        v = self.to_v(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(x.size(-1))
        # mask out padding tokens
        scores = scores.masked_fill(
            ~attention_mask.bool().unsqueeze(1), float("-1e9")
        )
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, v)
        x = self.norm1(x + attn_out)

        # Feed-forward
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)

        # Mean-pool over valid tokens instead of CLS token
        mask = attention_mask.unsqueeze(-1).type_as(x)
        summed = (x * mask).sum(1)
        lengths = mask.sum(1).clamp(min=1e-9)
        pooled = summed / lengths

        return self.classifier(pooled)

# 4️⃣ TRAINING SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerClassifier(vocab_size=vocab_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-2)

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)

# LR warmup schedule
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=500, num_training_steps=total_steps
)

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        mask      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}: Train Loss = {total_loss/len(train_loader):.4f}")

# 5️⃣ EVALUATION
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        mask      = batch["attention_mask"].to(device)
        labels    = batch["label"].to(device)

        logits = model(input_ids, mask)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
print(f"SST-2 Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/50.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1/3: Train Loss = 0.6791
Epoch 2/3: Train Loss = 0.6591
Epoch 3/3: Train Loss = 0.6499
SST-2 Validation → Accuracy: 0.6227, F1: 0.6126


### 🧪 Approach 3: Quantization-Aware Training (QAT) with Mixed Precision & STE

This approach introduces a more advanced **Quantization-Aware Training** pipeline using a custom BitLinear module that supports both:
- **1-bit quantization with Straight-Through Estimator (STE)**
- **Multi-bit fake quantization (MoQ)** with dynamic precision scheduling.

Key highlights:

- **Text Cleaning & Tokenization**: Similar to prior approaches, with negation handling and emoji demojization. Tokenized using BitNet's pretrained tokenizer.
- **BitLinearQAT Module**: A custom linear layer that applies dynamic quantization logic based on the training schedule. 1-bit uses STE; higher bits simulate precision using fake quantization.
- **Transformer Architecture**: Includes dropout for regularization, custom self-attention using `BitLinearQAT`, feedforward layers, and mean pooling for output.
- **Mixed Precision Schedule (MoQ)**: The bit-width used in quantization **linearly decays** from 8 bits to 1 bit across epochs, simulating a progressive constraint toward real quantized deployment.
- **Training Loop**: Incorporates warmup-based learning rate scheduling, parameter grouping for weight decay, and gradient clipping for stability.
- **Evaluation**: Final evaluation is performed with 1-bit quantization to simulate real deployment accuracy.




In [6]:
import re
import contractions
import emoji
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

# 1️⃣ DATA LOADING & CLEANING
raw = load_dataset("glue", "sst2")

def clean_text(example):
    sentence = example["sentence"]
    sentence = contractions.fix(sentence.lower())
    sentence = emoji.demojize(sentence)
    sentence = re.sub(r"[^a-zA-Z0-9\s!?]", "", sentence)
    sentence = re.sub(r"\bnot\s+(\w+)", r"not_\1", sentence)
    example["sentence"] = " ".join(sentence.split())
    return example

raw = raw.map(clean_text)

# 2️⃣ TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab_size = len(tokenizer)

def tokenize_fn(batch):
    return tokenizer(
        batch["sentence"], padding="max_length",
        truncation=True, max_length=128
    )

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3️⃣ BITLINEAR WITH STE & MIXED PRECISION
class BitLinearQAT(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, bit_width=1):
        # Mixed-precision / MoQ schedule
        if bit_width == 1:
            # 1-bit quantization with STE
            scale = self.weight.abs().mean(dim=1, keepdim=True)
            w_bin = scale * self.weight.sign()
            out = F.linear(x, w_bin, self.bias)
        else:
            # Fake quantization for higher bits
            max_val = self.weight.abs().max()
            q_max = 2**(bit_width-1) - 1
            scale = max_val / q_max if q_max>0 else 1.0
            w_q = torch.clamp((self.weight/scale).round(), -q_max, q_max) * scale
            out = F.linear(x, w_q, self.bias)
        return out

# 4️⃣ MODEL DEFINITION WITH DROPOUT & QAT
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class QuantizedTransformerQAT(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048, num_labels=2, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)
        self.attn_q = BitLinearQAT(embed_dim, embed_dim)
        self.attn_k = BitLinearQAT(embed_dim, embed_dim)
        self.attn_v = BitLinearQAT(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff1 = BitLinearQAT(embed_dim, ff_dim)
        self.ff2 = BitLinearQAT(ff_dim, embed_dim)
        self.ffn_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask, bit_width=1):
        x = self.embed(input_ids)
        x = self.pos_enc(x)

        # Self-attention
        Q = self.attn_q(x, bit_width)
        K = self.attn_k(x, bit_width)
        V = self.attn_v(x, bit_width)
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(x.size(-1))
        scores = scores.masked_fill(~attention_mask.bool().unsqueeze(1), float("-1e9"))
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, V)
        attn_out = self.attn_dropout(attn_out)
        x = self.norm1(x + attn_out)

        # Feed-forward
        ff_out = self.ff2(F.gelu(self.ff1(x, bit_width)), bit_width)
        ff_out = self.ffn_dropout(ff_out)
        x = self.norm2(x + ff_out)

        # Mean pooling
        mask = attention_mask.unsqueeze(-1).type_as(x)
        summed = (x * mask).sum(1)
        lengths = mask.sum(1).clamp(min=1e-9)
        pooled = summed / lengths

        return self.classifier(pooled)

# 5️⃣ TRAINING SETUP WITH MoQ SCHEDULE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerQAT(vocab_size=vocab_size).to(device)

# Parameter groups: no decay on embeddings & pos_enc
no_decay = ["embed", "pos_enc.pe", "bias", "LayerNorm.weight"]
optimizer_grouped = [
    {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 1e-2},
    {"params": [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
]
optimizer = optim.AdamW(optimizer_grouped, lr=2e-5)

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)
epochs = 5
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)

for epoch in range(epochs):
    # linearly decay bit width from 8→1
    bit_width = max(1, int(round(8 - 7 * (epoch / (epochs-1)))))

    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask, bit_width)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} [Bit={bit_width}] Train Loss: {total_loss/len(train_loader):.4f}")

# 6️⃣ EVALUATION
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(ids, mask, bit_width=1)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
print(f"Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1/5 [Bit=8] Train Loss: 0.6864
Epoch 2/5 [Bit=6] Train Loss: 0.6620
Epoch 3/5 [Bit=4] Train Loss: 0.6483
Epoch 4/5 [Bit=3] Train Loss: 0.6403
Epoch 5/5 [Bit=1] Train Loss: 0.6411
Validation → Accuracy: 0.6388, F1: 0.6365


### 🧪 Approach 4: Median-Scaled Quantization with STE and Custom BitLinear Layers

This final approach builds a highly efficient quantized model using **custom `BitLinearQAT` layers** enhanced with:

- **Straight-Through Estimator (STE)**: A custom autograd function allows binary weights to retain gradient flow during training, enabling 1-bit learning without breaking backpropagation.
- **Median-based Scaling**: Instead of mean or max, median scaling stabilizes weight quantization, especially for small or skewed distributions. This improves robustness, inspired by recent advances in low-bit LLMs.
- **Mixed-Precision Quantization (MoQ)**: A scheduled training strategy linearly decays bit-width from 8 to 1 across epochs, simulating gradual quantization.

Key features of this setup:

- **Text Preprocessing & Tokenization**: Cleaned SST-2 dataset using standard NLP techniques. BitNet tokenizer used for compatibility.
- **Transformer Architecture**: Includes quantized self-attention, GELU activations, dropout, and mean pooling for final representation.
- **Optimizer Strategy**: Weight decay is selectively applied, and a slightly higher learning rate is used for improved 1-bit convergence.
- **Evaluation**: Final testing is done at 1-bit precision to match deployment conditions. Accuracy and F1-score reflect the model's real-world viability.



In [7]:
import re
import contractions
import emoji
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

# Custom STE for smoother gradient flow (Sigmoid-STE or straight-through)
class STEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # forward uses sign, but gradient is passed through unchanged
        return x.sign()

    @staticmethod
    def backward(ctx, grad_output):
        # straight-through estimator: pass gradient directly
        return grad_output

# 1️⃣ DATA LOADING & CLEANING
raw = load_dataset("glue", "sst2")

def clean_text(example):
    sent = example["sentence"]
    sent = contractions.fix(sent.lower())
    sent = emoji.demojize(sent)
    sent = re.sub(r"[^a-zA-Z0-9\s!?]", "", sent)
    sent = re.sub(r"\bnot\s+(\w+)", r"not_\1", sent)
    example["sentence"] = " ".join(sent.split())
    return example

raw = raw.map(clean_text)

# 2️⃣ TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab_size = len(tokenizer)

def tokenize_fn(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3️⃣ BITLINEAR WITH MEDIAN SCALING & STE
class BitLinearQAT(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, bit_width=1):
        if bit_width == 1:
            # median-based scaling improves robustness on small data
            med = self.weight.abs().median(dim=1, keepdim=True).values
            w_bin = med * STEFunction.apply(self.weight)
            return F.linear(x, w_bin, self.bias)
        else:
            # fake-quant for smoother MoQ schedule
            maxv = self.weight.abs().max()
            qmax = 2**(bit_width-1) - 1
            scale = maxv / qmax if qmax>0 else 1.0
            w_q = torch.clamp((self.weight/scale).round(), -qmax, qmax) * scale
            return F.linear(x, w_q, self.bias)

# 4️⃣ MODEL DEFINITION WITH DROPOUT & QAT
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class QuantizedTransformerQAT(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048, num_labels=2, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)
        self.attn_q = BitLinearQAT(embed_dim, embed_dim)
        self.attn_k = BitLinearQAT(embed_dim, embed_dim)
        self.attn_v = BitLinearQAT(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff1 = BitLinearQAT(embed_dim, ff_dim)
        self.ff2 = BitLinearQAT(ff_dim, embed_dim)
        self.ffn_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask, bit_width=1):
        x = self.embed(input_ids)
        x = self.pos_enc(x)

        # Self-attention
        Q = self.attn_q(x, bit_width)
        K = self.attn_k(x, bit_width)
        V = self.attn_v(x, bit_width)
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(x.size(-1))
        scores = scores.masked_fill(~attention_mask.bool().unsqueeze(1), float("-1e9"))
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, V)
        attn_out = self.attn_dropout(attn_out)
        x = self.norm1(x + attn_out)

        # Feed-forward
        ff_out = self.ff2(F.gelu(self.ff1(x, bit_width)), bit_width)
        ff_out = self.ffn_dropout(ff_out)
        x = self.norm2(x + ff_out)

        # Mean pooling
        mask = attention_mask.unsqueeze(-1).type_as(x)
        summed = (x * mask).sum(1)
        lengths = mask.sum(1).clamp(min=1e-9)
        pooled = summed / lengths

        return self.classifier(pooled)

# 5️⃣ TRAINING SETUP WITH MoQ SCHEDULE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerQAT(vocab_size=vocab_size).to(device)

# Optimizer: freeze weight decay on embeddings & PE
no_decay = ["embed", "pos_enc.pe", "bias", "LayerNorm.weight"]
optimizer_grouped = [
    {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 1e-3},
    {"params": [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)],    "weight_decay": 0.0}
]
optimizer = optim.AdamW(optimizer_grouped, lr=3e-5)  # slightly higher LR helps binary training

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)
epochs = 7
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=len(train_loader)*epochs)

for epoch in range(epochs):
    bit_width = max(1, int(round(8 - 7 * (epoch/(epochs-1)))))
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids   = batch["input_ids"].to(device)
        mask  = batch["attention_mask"].to(device)
        labels= batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask, bit_width)
        loss   = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} [Bit={bit_width}] Loss={total_loss/len(train_loader):.4f}")

# 6️⃣ EVALUATION
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids    = batch["input_ids"].to(device)
        mask   = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(ids, mask, bit_width=1)
        preds  = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
print(f"Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1/7 [Bit=8] Loss=0.6799
Epoch 2/7 [Bit=7] Loss=0.6489
Epoch 3/7 [Bit=6] Loss=0.6280
Epoch 4/7 [Bit=4] Loss=0.6107
Epoch 5/7 [Bit=3] Loss=0.6002
Epoch 6/7 [Bit=2] Loss=0.6067
Epoch 7/7 [Bit=1] Loss=0.5714
Validation → Accuracy: 0.6984, F1: 0.6965


### 🧪 Approach 5: BitLinear with Median Scaling + STE + MoQ (7-Bit to 1-Bit Schedule)

This advanced quantization strategy builds on previous QAT pipelines with **three key enhancements**:

- **Custom STE Function**: A straight-through estimator (STE) passes gradients during backpropagation even for non-differentiable binary weights, allowing 1-bit training to remain stable.
- **Median-Based Scaling**: Instead of using mean or max scaling for quantization, median scaling is applied to the weights—boosting robustness on small or skewed datasets.
- **MoQ (Mixed-Precision Quantization) Schedule**: Bit-width is linearly decayed from 8 to 1 over training epochs. This enables gradual adaptation to extreme quantization without hurting early learning dynamics.

Model architecture is similar to previous approaches, featuring:
- Learned embeddings and positional encodings.
- Self-attention and feed-forward layers built entirely with `BitLinearQAT`.
- Dropout and residual layer normalization.
- Mean pooling instead of a [CLS] token for final classification.




In [10]:
import re
import contractions
import emoji
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

# Custom STE for smoother gradient flow (Sigmoid-STE or straight-through)
class STEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # forward uses sign, but gradient is passed through unchanged
        return x.sign()

    @staticmethod
    def backward(ctx, grad_output):
        # straight-through estimator: pass gradient directly
        return grad_output

# 1️⃣ DATA LOADING & CLEANING
raw = load_dataset("glue", "sst2")

def clean_text(example):
    sent = example["sentence"]
    sent = contractions.fix(sent.lower())
    sent = emoji.demojize(sent)
    sent = re.sub(r"[^a-zA-Z0-9\s!?]", "", sent)
    sent = re.sub(r"\bnot\s+(\w+)", r"not_\1", sent)
    example["sentence"] = " ".join(sent.split())
    return example

raw = raw.map(clean_text)

# 2️⃣ TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab_size = len(tokenizer)

def tokenize_fn(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3️⃣ BITLINEAR WITH MEDIAN SCALING & STE
class BitLinearQAT(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, bit_width=1):
        if bit_width == 1:
            # median-based scaling improves robustness on small data
            med = self.weight.abs().median(dim=1, keepdim=True).values
            w_bin = med * STEFunction.apply(self.weight)
            return F.linear(x, w_bin, self.bias)
        else:
            # fake-quant for smoother MoQ schedule
            maxv = self.weight.abs().max()
            qmax = 2**(bit_width-1) - 1
            scale = maxv / qmax if qmax > 0 else 1.0
            w_q = torch.clamp((self.weight/scale).round(), -qmax, qmax) * scale
            return F.linear(x, w_q, self.bias)

# 4️⃣ MODEL DEFINITION WITH DROPOUT & QAT
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

class QuantizedTransformerQAT(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048, num_labels=2, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)
        self.attn_q = BitLinearQAT(embed_dim, embed_dim)
        self.attn_k = BitLinearQAT(embed_dim, embed_dim)
        self.attn_v = BitLinearQAT(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff1 = BitLinearQAT(embed_dim, ff_dim)
        self.ff2 = BitLinearQAT(ff_dim, embed_dim)
        self.ffn_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask, bit_width=1):
        x = self.embed(input_ids)
        x = self.pos_enc(x)

        # Self-attention
        Q = self.attn_q(x, bit_width)
        K = self.attn_k(x, bit_width)
        V = self.attn_v(x, bit_width)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(x.size(-1))
        scores = scores.masked_fill(~attention_mask.bool().unsqueeze(1), float("-1e9"))
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, V)
        attn_out = self.attn_dropout(attn_out)
        x = self.norm1(x + attn_out)

        # Feed-forward
        ff_out = self.ff2(F.gelu(self.ff1(x, bit_width)), bit_width)
        ff_out = self.ffn_dropout(ff_out)
        x = self.norm2(x + ff_out)

        # Mean pooling
        mask = attention_mask.unsqueeze(-1).type_as(x)
        summed = (x * mask).sum(1)
        lengths = mask.sum(1).clamp(min=1e-9)
        pooled = summed / lengths

        return self.classifier(pooled)

# 5️⃣ TRAINING SETUP WITH MoQ SCHEDULE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerQAT(vocab_size=vocab_size).to(device)

# Optimizer: freeze weight decay on embeddings & PE
no_decay = ["embed", "pos_enc.pe", "bias", "LayerNorm.weight"]
optimizer_grouped = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 1e-3},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
]
optimizer = optim.AdamW(optimizer_grouped, lr=3e-5)

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader = DataLoader(data["validation"], batch_size=64)
epochs = 7
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=50, num_training_steps=len(train_loader) * epochs
)

for epoch in range(epochs):
    bit_width = max(1, int(round(8 - 7 * (epoch / (epochs - 1)))))
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask, bit_width)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} [Bit={bit_width}] Loss={total_loss/len(train_loader):.4f}")

# 6️⃣ EVALUATION
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(ids, mask, bit_width=1)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
print(f"Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Epoch 1/7 [Bit=8] Loss=0.6685
Epoch 2/7 [Bit=7] Loss=0.6396
Epoch 3/7 [Bit=6] Loss=0.6168
Epoch 4/7 [Bit=4] Loss=0.5991
Epoch 5/7 [Bit=3] Loss=0.5887
Epoch 6/7 [Bit=2] Loss=0.6053
Epoch 7/7 [Bit=1] Loss=0.5611
Validation → Accuracy: 0.7076, F1: 0.7066


### 🚀 Approach 6: Multi-Head Quantized Transformer with Stacked Layers and Progressive Bit Decay

This approach builds the **most advanced quantized model** in the series by integrating deep transformer stacking, multi-head self-attention, and dynamic quantization.

#### 🧩 Key Components:

- **Custom BitLinearQAT Layer**: Supports both 1-bit quantization (via STE + median scaling) and fake multi-bit quantization. Enables progressive precision decay during training.
- **STE (Straight-Through Estimator)**: Allows backpropagation through non-differentiable binary activations—essential for training with extreme quantization.
- **Positional Encoding**: Classic sinusoidal encoding ensures token position awareness across transformer layers.
- **Multi-Head Attention Block**: Implements scaled dot-product attention with multiple heads using quantized projections (Q, K, V), followed by dropout, residuals, and layer normalization.
- **Stacked Transformer Layers**: Multiple transformer blocks are stacked using `nn.ModuleList` to increase depth and expressiveness.
- **MoQ Schedule**: The bit-width used for quantization decays linearly from 8 to 1 over the training epochs, helping the model adapt gradually to extreme compression.
- **Mean Pooling Output**: Pooled token representations are used for classification, avoiding reliance on special tokens like `[CLS]`.

#### 🛠️ Training Details:

- Optimizer: AdamW with selective weight decay exclusion.
- Scheduler: Warmup followed by linear decay.
- Stability: Includes gradient clipping and dropout regularization.




In [12]:
import re
import contractions
import emoji
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

# Custom STE for smoother gradient flow
class STEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x.sign()
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

# 1️⃣ DATA LOADING & CLEANING
raw = load_dataset("glue", "sst2")

def clean_text(example):
    sent = example["sentence"]
    sent = contractions.fix(sent.lower())
    sent = emoji.demojize(sent)
    sent = re.sub(r"[^a-zA-Z0-9\s!?]", "", sent)
    sent = re.sub(r"\bnot\s+(\w+)", r"not_\1", sent)
    example["sentence"] = " ".join(sent.split())
    return example

raw = raw.map(clean_text)

# 2️⃣ TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/bitnet-b1.58-2B-4T", trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
vocab_size = len(tokenizer)

def tokenize_fn(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

data = raw.map(tokenize_fn, batched=True)
data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# 3️⃣ BITLINEAR WITH MEDIAN SCALING & STE
class BitLinearQAT(nn.Module):
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, bit_width=1):
        if bit_width == 1:
            med = self.weight.abs().median(dim=1, keepdim=True).values
            w_bin = med * STEFunction.apply(self.weight)
            return F.linear(x, w_bin, self.bias)
        else:
            maxv = self.weight.abs().max()
            qmax = 2**(bit_width-1) - 1
            scale = maxv / qmax if qmax > 0 else 1.0
            w_q = torch.clamp((self.weight/scale).round(), -qmax, qmax) * scale
            return F.linear(x, w_q, self.bias)

# 4️⃣ POSITIONAL ENCODING
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)
    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)

# 5️⃣ TRANSFORMER BLOCK WITH MULTI-HEAD
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.num_heads = num_heads
        head_dim = embed_dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = BitLinearQAT(embed_dim, embed_dim * 3)
        self.out_proj = BitLinearQAT(embed_dim, embed_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff1 = BitLinearQAT(embed_dim, ff_dim)
        self.ff2 = BitLinearQAT(ff_dim, embed_dim)
        self.ff_dropout = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x, mask, bit_width):
        B, T, D = x.size()
        q, k, v = self.qkv(x, bit_width).chunk(3, dim=-1)
        # reshape for multi-head
        q = q.view(B, T, self.num_heads, D//self.num_heads).permute(0,2,1,3)
        k = k.view(B, T, self.num_heads, D//self.num_heads).permute(0,2,1,3)
        v = v.view(B, T, self.num_heads, D//self.num_heads).permute(0,2,1,3)
        # scaled dot-product
        scores = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        # mask padding tokens on keys
        mask_k = mask.unsqueeze(1).unsqueeze(2)  # (B,1,1,T)
        scores = scores.masked_fill(~mask_k, float('-1e9'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        context = torch.matmul(attn, v)
        # combine heads
        context = context.permute(0,2,1,3).contiguous().view(B, T, D)
        out = self.out_proj(context, bit_width)
        out = self.attn_dropout(out)
        x = self.norm1(x + out)
        # feed-forward
        ff = self.ff2(F.gelu(self.ff1(x, bit_width)), bit_width)
        ff = self.ff_dropout(ff)
        x = self.norm2(x + ff)
        return x

# 6️⃣ MODEL DEFINITION WITH STACKED LAYERS
class QuantizedTransformerQAT(nn.Module):
    def __init__(self, vocab_size, embed_dim=768, num_heads=8, ff_dim=2048,
                 num_layers=2, num_labels=2, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_enc = PositionalEncoding(embed_dim)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(embed_dim, num_labels)

    def forward(self, input_ids, attention_mask, bit_width=1):
        x = self.embed(input_ids)
        x = self.pos_enc(x)
        mask = attention_mask.bool()
        for layer in self.layers:
            x = layer(x, mask, bit_width)
        mask = mask.unsqueeze(-1)
        summed = (x * mask).sum(1)
        lengths = mask.sum(1).clamp(min=1e-9)
        pooled = summed / lengths
        return self.classifier(pooled)

# 7️⃣ TRAINING SETUP WITH PROGRESSIVE MoQ
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = QuantizedTransformerQAT(vocab_size=vocab_size).to(device)
no_decay = ["embed", "pos_enc.pe", "bias", "LayerNorm.weight"]
optimizer_grouped = [
    {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": 1e-3},
    {"params": [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}
]
optimizer = optim.AdamW(optimizer_grouped, lr=3e-5)
train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)
epochs = 7
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=total_steps)

for epoch in range(epochs):
    bit_width = max(1, int(round(8 - 7 * (epoch / (epochs - 1)))))
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        logits = model(ids, mask, bit_width)
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} [Bit={bit_width}] Loss={total_loss/len(train_loader):.4f}")

# 8️⃣ EVALUATION
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(ids, mask, bit_width=1)
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="weighted")
print(f"Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


Epoch 1/7 [Bit=8] Loss=0.6762
Epoch 2/7 [Bit=7] Loss=0.6468
Epoch 3/7 [Bit=6] Loss=0.6255
Epoch 4/7 [Bit=4] Loss=0.6085
Epoch 5/7 [Bit=3] Loss=0.5989
Epoch 6/7 [Bit=2] Loss=0.6391
Epoch 7/7 [Bit=1] Loss=0.5618
Validation → Accuracy: 0.7018, F1: 0.7015


### 🧠 Approach 7: BitLinear Quantization-Aware Fine-Tuning of Pretrained BERT

This approach applies quantization-aware training (QAT) directly to a **pretrained BERT model** by replacing all `nn.Linear` layers with custom `BitLinearQAT` layers. It fine-tunes the quantized model end-to-end on the SST-2 sentiment classification task.

#### 🔧 Key Components:

- **Custom BitLinearQAT Layer**: Replaces standard linear layers with quantized versions that support 1-bit precision using **median-based scaling and STE** for gradient propagation.
- **Layer Swapping**: All `nn.Linear` instances in the BERT model are recursively swapped with `BitLinearQAT` while retaining pretrained weights.
- **Data Cleaning & Tokenization**: Input sentences are normalized (lowercased, emoji-demojized, contractions expanded), then tokenized using `bert-base-uncased`.
- **Training Regimen**:
  - Fine-tuning is done in full 1-bit quantization mode.
  - Gradient clipping and warmup-based learning rate scheduling are applied for stability.
  - Uses AdamW optimizer with weight decay grouping.

#### 🧪 Evaluation:

The model is evaluated using **accuracy** and **weighted F1-score** on the SST-2 validation split, simulating inference under binary-weight constraints.


In [4]:
import re
import contractions
import emoji
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

# Custom STE for smoother gradient flow
def STEFunction(x):
    # Straight-through estimator wrapper
    return x.sign()

class STELayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        return x.sign()
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

# 1️⃣ LOAD PRETRAINED BERT CLASSIFIER
model_name = "bert-base-uncased"
teacher = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher.to(device)

# 2️⃣ DEFINE BitLinearQAT
class BitLinearQAT(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        nn.init.xavier_uniform_(self.weight)
        if self.bias is not None:
            nn.init.zeros_(self.bias)

    def forward(self, x, bit_width=1):
        if bit_width == 1:
            # median scaling + STE
            scale = self.weight.abs().median(dim=1, keepdim=True).values
            wq = scale * STELayer.apply(self.weight)
        else:
            # fake quant for higher bits
            maxv = self.weight.abs().max()
            qmax = 2**(bit_width-1) - 1
            scale = maxv / qmax if qmax>0 else 1.0
            w_round = torch.clamp((self.weight/scale).round(), -qmax, qmax)
            wq = w_round * scale
        return F.linear(x, wq, self.bias)

# 3️⃣ SWAP LINEARS IN BERT
import torch.nn as nn

def replace_linears(module):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            qlin = BitLinearQAT(child.in_features, child.out_features, bias=(child.bias is not None))
            # copy weights
            qlin.weight.data.copy_(child.weight.data)
            if child.bias is not None:
                qlin.bias.data.copy_(child.bias.data)
            setattr(module, name, qlin)
        else:
            replace_linears(child)

replace_linears(teacher)
# ensure all quantized layers are on the correct device
teacher.to(device)
model = teacher  # quantized model ready

# 4️⃣ DATA LOADING & CLEANING
def clean_text(sent):
    sent = contractions.fix(sent.lower())
    sent = emoji.demojize(sent)
    sent = re.sub(r"[^a-zA-Z0-9\s!?]", "", sent)
    sent = re.sub(r"\bnot\s+(\w+)", r"not_\1", sent)
    return " ".join(sent.split())

raw = load_dataset("glue", "sst2")
raw = raw.map(lambda ex: {"sentence": clean_text(ex["sentence"]), "label": ex["label"]})

# 5️⃣ TOKENIZATION
def tokenize_fn(batch):
    return tokenizer(batch["sentence"], padding="max_length", truncation=True, max_length=128)

data = raw.map(tokenize_fn, batched=True)
for split in data:
    data[split] = data[split].with_format(type="torch", columns=["input_ids","attention_mask","label"])

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)

# 6️⃣ OPTIMIZER & SCHEDULER
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped = [
    {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay":1e-2},
    {"params": [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay":0.0}
]
optimizer = optim.AdamW(optimizer_grouped, lr=3e-5)
epochs = 3
total_steps = len(train_loader)*epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=50, num_training_steps=total_steps)

# 7️⃣ TRAINING LOOP (1-bit QAT)
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        inputs = batch["input_ids"].to(device)
        masks  = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs, attention_mask=masks, labels=None)
        logits = outputs.logits
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} Loss: {total_loss/len(train_loader):.4f}")

# 8️⃣ EVALUATION
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        inputs = batch["input_ids"].to(device)
        masks  = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        logits = model(inputs, attention_mask=masks).logits
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
print(f"Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1/3 Loss: 0.3150
Epoch 2/3 Loss: 0.1881
Epoch 3/3 Loss: 0.1289
Validation → Accuracy: 0.8567, F1: 0.8565


### 🔬 Approach 8: Full-Stack Quantized BERT with Ternary Weights, 8-bit Activations, and Custom Normalization

This approach introduces an aggressive yet structured quantization-aware training (QAT) setup applied directly to a pretrained **BERT-base** model.

#### 🧠 Core Innovations:

- **Ternary Weight Quantization**:
  - Replaces all `nn.Linear` layers with `TernaryBitLinearQAT`, where weights are quantized to {−scale, 0, +scale}.
  - Uses **Straight-Through Estimator (STE)** for enabling gradient flow through non-differentiable ternary weights.
  - Includes dynamic masking to zero out insignificant weights for effective sparsity.

- **8-bit Activation Quantization**:
  - Post-layer outputs are quantized to simulate **int8 inference** during training.
  - Ensures activations remain deployment-friendly and hardware-efficient.

- **ReLU → ReLU² & LayerNorm → SubLayerNorm**:
  - Replaces ReLU with **ReLU²**, often used in energy-efficient models.
  - Standard `LayerNorm` is replaced by **SubLayerNorm**, which removes the bias term while preserving scale (`gamma`)—streamlining normalization.

- **Module Replacement Utility**:
  - A recursive function automatically swaps linear, activation, and normalization layers with their quantized counterparts, while preserving pretrained weights.

#### ⚙️ Training:

- SST-2 dataset is cleaned and tokenized using `bert-base-uncased`.
- Optimizer uses grouped parameter decay, and training employs a warmup-based scheduler.
- Full QAT is performed over 3 epochs using clipped gradients and AdamW.

#### 📈 Evaluation:

- Final predictions are made with fully quantized weights and activations.
- **Accuracy** and **Weighted F1-score** are computed on the SST-2 validation set.



In [5]:
import re
import contractions
import emoji
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

# ─── STE & Quantization Helpers ────────────────────────────────────────────────

class STELayer(torch.autograd.Function):
    """Straight‐Through Estimator for sign()."""
    @staticmethod
    def forward(ctx, x):
        return x.sign()
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

def quantize_activation(x, num_bits=8):
    """Per‐tensor absmax quantization to signed ints and back to float."""
    qmax = 2**(num_bits - 1) - 1
    max_val = x.abs().max()
    scale = max_val / qmax if max_val != 0 else 1.0
    xq = torch.clamp((x / scale).round(), -qmax, qmax)
    return xq * scale

# ─── Custom QAT Layers ────────────────────────────────────────────────────────

class TernaryBitLinearQAT(nn.Module):
    """
    Linear layer with ternary weight quantization (−scale, 0, +scale via STE)
    and 8-bit activation quantization.
    """
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        # no bias (SubLN handles scale only)
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        # ▪️ compute scale = mean(|w|)
        scale = self.weight.abs().mean(dim=1, keepdim=True)
        # ▪️ sign(w) with STE backward
        sign_w = STELayer.apply(self.weight)
        # ▪️ mask to zero out small weights (ternary)
        mask = (self.weight.abs() >= scale).float()
        # ▪️ quantized weights in {−scale, 0, +scale}
        wq = sign_w * scale * mask
        out = F.linear(x, wq, bias=None)
        # ▪️ quantize activations to 8-bit then rescale
        return quantize_activation(out)

class ReLUSquared(nn.Module):
    """Squared ReLU activation (ReLU(x)^2)."""
    def forward(self, x):
        return F.relu(x).pow(2)

class SubLayerNorm(nn.Module):
    """Sub-Layer Normalization (no bias term)."""
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(normalized_shape))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std  = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps)

# ─── Module Replacement Utility ───────────────────────────────────────────────

def replace_modules(module):
    """
    Recursively replace:
      - nn.Linear   → TernaryBitLinearQAT (copying pretrained weights)
      - nn.ReLU     → ReLUSquared
      - nn.LayerNorm→ SubLayerNorm (copying gamma from LayerNorm.weight)
    """
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            # create QAT layer and copy pretrained weights
            qlin = TernaryBitLinearQAT(child.in_features, child.out_features)
            with torch.no_grad():
                qlin.weight.copy_(child.weight.data)
            setattr(module, name, qlin)

        elif isinstance(child, nn.ReLU):
            setattr(module, name, ReLUSquared())

        elif isinstance(child, nn.LayerNorm):
            subln = SubLayerNorm(child.normalized_shape, eps=child.eps)
            with torch.no_grad():
                # copy original LayerNorm scale (gamma)
                subln.gamma.copy_(child.weight.data)
            setattr(module, name, subln)

        else:
            replace_modules(child)

# ─── Load Pretrained BERT & Apply QAT Modifications ───────────────────────────

model_name = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(model_name)

replace_modules(model)  # apply quantized layers, activations, and normalization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# ─── Data Cleaning & Tokenization ──────────────────────────────────────────────

def clean_text(sent):
    sent = contractions.fix(sent.lower())
    sent = emoji.demojize(sent)
    sent = re.sub(r"[^a-zA-Z0-9\s!?]", "", sent)
    sent = re.sub(r"\bnot\s+(\w+)", r"not_\1", sent)
    return " ".join(sent.split())

raw = load_dataset("glue", "sst2")
raw = raw.map(lambda ex: {"sentence": clean_text(ex["sentence"]), "label": ex["label"]})

def tokenize_fn(batch):
    return tokenizer(batch["sentence"],
                     padding="max_length",
                     truncation=True,
                     max_length=128)

data = raw.map(tokenize_fn, batched=True)
for split in data:
    data[split] = data[split].with_format(type="torch",
                                          columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(data["train"], batch_size=32, shuffle=True)
val_loader   = DataLoader(data["validation"], batch_size=64)

# ─── Optimizer & Scheduler ─────────────────────────────────────────────────────

no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped = [
    {
      "params": [
        p for n, p in model.named_parameters()
        if not any(nd in n for nd in no_decay)
      ],
      "weight_decay": 1e-2
    },
    {
      "params": [
        p for n, p in model.named_parameters()
        if any(nd in n for nd in no_decay)
      ],
      "weight_decay": 0.0
    },
]
optimizer = optim.AdamW(optimizer_grouped, lr=3e-5)

epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=50,
    num_training_steps=total_steps
)

# ─── Training Loop ─────────────────────────────────────────────────────────────

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        inputs = batch["input_ids"].to(device)
        masks  = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs, attention_mask=masks)
        logits  = outputs.logits
        loss    = F.cross_entropy(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}  Loss: {avg_loss:.4f}")

# ─── Evaluation ────────────────────────────────────────────────────────────────

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        inputs = batch["input_ids"].to(device)
        masks  = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(inputs, attention_mask=masks).logits
        preds  = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average="weighted")
print(f"Validation → Accuracy: {acc:.4f}, F1: {f1:.4f}")


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Epoch 1/3  Loss: 0.6930
Epoch 2/3  Loss: 0.6909
Epoch 3/3  Loss: 0.6870
Validation → Accuracy: 0.5092, F1: 0.3436
